# Orchestration patterns

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/concepts/orchestration-patterns.ipynb)

An orchestrator coordinates dependencies, retries and recovery. The task implementation still owns idempotency and data correctness. Choose between schedule-driven batches, event-triggered work and continuous processing based on the freshness contract.

## Decisions and trade-offs

Give each logical batch a stable identity. Retry only transient failures, use bounded backoff and avoid publishing before quality gates pass. Keep backfills on the same transformation path as scheduled runs. Prefer a durable record of completion over an in-memory flag.

The exercise models a publish ledger; it is not a durable scheduler. Crash consistency requires a transactional ledger or an idempotent destination.

[Airflow best practices](https://airflow.apache.org/docs/apache-airflow/stable/best-practices.html)

## Worked example

Run the small fixture and inspect each assertion. Change one input to see which contract fails.

In [ ]:
published={}
def publish(batch_id, rows):
    value=tuple(rows)
    if batch_id in published and published[batch_id]!=value:
        raise ValueError('same batch identity with different content')
    published[batch_id]=value
publish('2026-01-01',[1,2]); publish('2026-01-01',[1,2])
assert len(published)==1
try: publish('2026-01-01',[9])
except ValueError: pass
else: raise AssertionError('identity conflict not detected')

## Practice

Add a failure case, explain the expected outcome before running it, and identify which guarantee would need a real service or a larger dataset to verify.